# 📗 오픈소스 LLM을 API로 서비스하기

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<div style="background:#ffffff;color:#202124;font-size:17px;line-height:1.6;padding:6px 10px;border-left:4px solid #54769b;">
<strong style="color:#1f3a5f;font-size:19px;">실습 환경</strong><br>
<strong style="font-size:1.13em;color:#1f3a5f;">서버:</strong> RunPod의 Linux + NVIDIA GPU에서 실행합니다. 이 CUDA 서버 실습은 macOS·Windows 기본 환경에서 실행하지 않습니다.<br>
<strong style="font-size:1.13em;color:#1f3a5f;">클라이언트:</strong> 내 PC(Mac·Windows·Linux)의 VS Code에서 노트북을 실행해 RunPod의 vLLM API를 호출합니다. PC에는 GPU가 필요 없습니다.<br>
<strong style="font-size:1.13em;color:#1f3a5f;">최소 권장:</strong> GPU VRAM 16GB 1장 · RAM 16GB · 디스크 80GB. 모델 서버는 하나씩 실행합니다.
</div>
</div>

#### **vLLM**

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">LLM을 실행해 <strong style="font-size:1.13em;color:#1f3a5f;">여러 요청을 효율적으로 처리하고, OpenAI 호환 API로 제공하는 오픈소스 추론·서빙 엔진</strong>입니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">LLM 모델 서빙</strong>은 모델을 실행 상태로 유지하면서 API 요청에 답을 생성해 제공하는 일입니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">배우는 이유:</strong> 직접 학습하거나 선택한 모델을 여러 사용자가 이용할 수 있게 만듭니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">활용처:</strong> 사내 챗봇·고객 상담·RAG 서비스의 답변 생성 서버입니다.</p>
</div>

<img style="display:block;max-width:100%;height:auto;margin:0;" src="images/03_client_and_server.png" width="1050" alt="클라이언트가 API로 요청하면 LLM 서버가 모델을 실행해 답변을 돌려줍니다.">

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">저장한 노무 상담 모델을 <strong style="font-size:1.13em;color:#1f3a5f;">RunPod에서 vLLM 서버로 실행</strong>하고, <strong style="font-size:1.13em;color:#1f3a5f;">내 PC에서 공개 HTTPS 주소로 질문</strong>합니다. 답변을 받은 뒤 메모리와 여러 요청을 처리하는 원리를 살펴봅니다.</p>
</div>

<img style="display:block;max-width:100%;height:auto;margin:0;" src="images/00_lesson01_goals.png" width="1050" alt="학습 목표: 서빙 구조, 서버 실행, API 요청, 여러 요청 처리.">

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">학습 목표</strong> (그림의 번호와 같습니다)</p>
<ol style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">클라이언트·서버의 역할과 모델 서빙의 의미를 설명합니다.</li>
<li style="margin:3px 0;">터미널에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">vllm serve</code>로 모델 서버를 실행합니다.</li>
<li style="margin:3px 0;">OpenAI 호환 API로 질문하고 답변과 종료 이유를 확인합니다.</li>
<li style="margin:3px 0;">PagedAttention·연속 배치의 역할을 설명하고 여러 요청을 보냅니다.</li>
</ol>
</div>

## 1. LLM 모델 서빙의 구조

### 1-1. 클라이언트·서버·API

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">클라이언트(client):</strong> 기능을 사용하기 위해 요청을 보내는 프로그램입니다. 웹·앱·에이전트 등이 해당합니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">서버(server):</strong> 요청을 받아 기능이나 결과를 제공하는 프로그램입니다. LLM 서버는 모델을 실행해 답을 생성합니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">API:</strong> 클라이언트와 서버가 요청과 응답을 주고받는 규칙입니다.</li>
</ul>
</div>

### 1-2. 직접 실행하는 LLM을 API로 제공하기

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">vLLM은 LLM을 직접 실행해 API로 제공하고 싶을 때 사용합니다.</strong> 내 PC나 RunPod 같은 GPU 서버에 모델을 올리고, 다른 프로그램이 OpenAI 호환 API로 질문할 수 있게 만듭니다.</p>
<p style="margin:6px 0;">이미 제공되는 외부 LLM API를 호출해 에이전트를 만드는 경우에는 vLLM이 필요하지 않습니다.</p>
</div>

### 1-3. 모델 서버의 실행 흐름

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">서버를 시작하면 <strong style="font-size:1.13em;color:#1f3a5f;">모델을 먼저 불러오고 요청을 기다립니다</strong>. 이후에는 요청을 받을 때마다 답을 생성하고 돌려줍니다. 질문마다 모델을 다시 불러오지 않습니다.</p>
</div>

<img style="display:block;max-width:100%;height:auto;margin:0;" src="images/06_model_server_lifecycle.png" width="1050" alt="서버 시작 때 모델을 불러오고, 요청 대기·질문 수신·답변 생성·응답 전달을 반복합니다.">

## 2. 실습 준비

### 2-1. 라이브러리와 데이터

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">이 노트북은 <strong style="font-size:1.13em;color:#1f3a5f;">내 PC에서 실행하는 클라이언트</strong>입니다. PC와 RunPod 모두에 과정 파일을 준비합니다. 질문은 PC의 SFT 평가 파일에서 읽고, 모델 가중치는 RunPod가 불러옵니다.</p>
<p style="margin:6px 0;">RunPod 환경과 접속 정보는 <a style="color:#245fa0;text-decoration:underline;" href="%EB%B6%80%EB%A1%9D_01_RunPod_%ED%99%98%EA%B2%BD%EA%B3%BC_%EC%99%B8%EB%B6%80_API_%EC%97%B0%EA%B2%B0.ipynb">부록 01</a>에서 준비합니다. 이 교안에서는 모델 준비부터 서버 실행·API 요청까지 진행합니다.</p>
</div>

### 2-2. 접속 설정과 평가 데이터 불러오기

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">내 PC의 노트북에서 서버 주소·API 키와 SFT 평가 질문을 불러옵니다. 환경변수 준비 다음 셀에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>의 URL과 API 키가 읽혔는지 확인합니다. API 키가 표시되므로 실행 결과가 남은 노트북은 공유하지 않습니다. 아래부터 Python 셀은 내 PC, 서버 명령은 RunPod 터미널에서 실행합니다.</p>
</div>

In [ ]:
import json
import os
import re
from pathlib import Path

import pandas as pd
from openai import OpenAI
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))
material_dir = Path(".").resolve()
output_dir = material_dir / "output"
output_dir.mkdir(exist_ok=True)
day70_dir = material_dir.parent / "day70_SFT_지도파인튜닝"
SERVER_URL = os.environ["VLLM_SERVER_URL"].rstrip("/")  # RunPod 공개 HTTPS 주소
VLLM_API_KEY = os.environ["VLLM_API_KEY"]
MAX_TOKENS = 256     # 답변으로 생성할 최대 토큰 수
pd.set_option("display.max_colwidth", None)

In [ ]:
print("VLLM_SERVER_URL:", SERVER_URL)
print("VLLM_API_KEY:", VLLM_API_KEY)

In [ ]:
def load_jsonl(path):
    """JSONL 파일을 한 줄에 딕셔너리 하나씩 읽어 리스트로 돌려줍니다(기존 실습과 같은 함수)."""
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

In [ ]:
SYSTEM_PROMPT = "너는 노무 상담 챗봇이다. 노동법 조문에 근거해 존댓말로 간결히 답한다."
labor_eval_rows = load_jsonl(day70_dir / "data" / "본실습" / "labor_eval.jsonl")

## 3. 터미널에서 모델 서버 실행하기

### 3-1. 모델과 서버 이름

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">SFT 단원에서 학습한 노무 상담 모델을 <strong style="font-size:1.13em;color:#1f3a5f;">허브에서 바로 불러옵니다</strong>. 다시 학습하지 않습니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">모델 저장소</strong>는 가중치를 받을 위치이고, <strong style="font-size:1.13em;color:#1f3a5f;">서버 이름 <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">labor-sft</code></strong>는 API 요청에서 사용할 이름입니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">직접 학습한 Qwen3.5 텍스트 모델을 올릴 때:</strong> 허브에 업로드하기 전에 아래처럼 저장합니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">save_original_format=False</code>는 가중치 이름을 텍스트 모델 구조에 맞춰 저장하는 옵션입니다. 이를 생략하면 vLLM에서 모델 구조와 가중치 이름이 맞지 않아 로드에 실패할 수 있습니다. 이 실습의 SFT 모델에는 이미 적용되어 있습니다.</p>
</div>

```python
model.save_pretrained(model_dir, save_original_format=False)
tokenizer.save_pretrained(model_dir)
```

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">부록 01에서 준비한 RunPod 터미널</strong>에서 아래 명령을 실행합니다. 준비한 환경변수 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">VLLM_API_KEY</code>를 서버 인증키로 지정합니다.</p>
</div>

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">명령어 설정</strong></p>
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:6px 0;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">설정</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">의미</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">uv run vllm serve &lt;모델&gt;</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">day75 폴더의 RunPod 전용 uv 환경에서 모델을 불러와 API 서버를 시작합니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--served-model-name</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">클라이언트가 요청의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">model</code>에 넣을 이름입니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--dtype bfloat16</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">모델 가중치와 추론 중간값을 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">bfloat16</code> 자료형으로 계산합니다. bfloat16은 16비트 BF16 형식입니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--max-model-len 4096</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">시스템 메시지·질문·답변을 합친 최대 토큰 수를 <strong style="font-size:1.13em;color:#1f3a5f;">4,096</strong>으로 제한합니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--gpu-memory-utilization</code> 생략</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">공식 기본값 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">0.92</code>로 GPU 메모리의 92%를 실행 예산으로 사용합니다. 명시 예: <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--gpu-memory-utilization 0.92</code>.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--reasoning-parser qwen3</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">Qwen이 출력한 생각 부분과 최종 답변을 구분합니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--default-chat-template-kwargs</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">enable_thinking=false</code>로 별도의 생각 생성을 끕니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--host 0.0.0.0</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">Pod 외부에서 들어오는 연결도 받습니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--port 8000</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">RunPod에서 열어 둔 8000번 포트로 요청을 받습니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--api-key &quot;$VLLM_API_KEY&quot;</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">환경변수에 저장한 키로 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/v1</code> API 요청을 인증합니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">tee</code> 로그 저장</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">서버 출력과 오류를 터미널과 로그 파일에 함께 남깁니다.</td>
</tr>
</tbody>
</table>
</div>

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">GPU 메모리 예산은 <a style="color:#245fa0;text-decoration:underline;" href="https://docs.vllm.ai/en/v0.27.0/configuration/engine_args/">vLLM 0.27.0 공식 기본값</a>인 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">0.92</code>를 사용합니다. 자료형·최대 길이·LoRA 용량은 각 명령과 표에 표시한 실습 설정을 따릅니다. 모델 선택·인증·외부 접속과 Qwen의 생각 모드 설정도 실습 동작에 맞춰 지정합니다.</p>
</div>

**복사해서 실행 · RunPod 터미널**

```bash
uv run vllm serve muahseol/enkoa-qwen3.5-0.8b-labor-sft \
  --served-model-name labor-sft \
  --dtype bfloat16 \
  --max-model-len 4096 \
  --reasoning-parser qwen3 \
  --default-chat-template-kwargs '{"enable_thinking": false}' \
  --host 0.0.0.0 \
  --port 8000 \
  --api-key "$VLLM_API_KEY" 2>&1 | tee output/labor_server.log
```

### 3-2. 서버 준비 확인

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">로그에 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">Application startup complete</code></strong>가 나오면 요청을 보낼 수 있습니다. RunPod 터미널은 그대로 두고 <strong style="font-size:1.13em;color:#1f3a5f;">내 PC의 노트북</strong>으로 돌아옵니다. 첫 실행은 모델 다운로드와 GPU 계산 준비에 시간이 걸릴 수 있습니다.</p>
</div>

## 4. OpenAI 호환 API로 질문하기

### 4-1. 클라이언트 연결과 모델 목록

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">OpenAI 호환 API</strong>는 OpenAI SDK로 요청할 수 있는 API입니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">base_url</code>에 vLLM 서버 주소를 지정하면 같은 SDK로 직접 실행한 모델을 사용할 수 있습니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">내 PC → RunPod의 공개 HTTPS 주소 → vLLM</strong>으로 요청합니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">localhost</code>는 지금 코드를 실행하는 PC를 가리키므로 사용하지 않습니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">0.0.0.0</code>도 클라이언트의 접속 주소가 아닙니다.</p>
<p style="margin:6px 0;">SDK의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">api_key</code>에 서버와 같은 실습용 키를 넣습니다. API 형식은 <a style="color:#245fa0;text-decoration:underline;" href="https://docs.vllm.ai/en/v0.27.0/serving/openai_compatible_server/">vLLM 공식 문서</a>에서 확인할 수 있습니다.</p>
</div>

In [ ]:
client = OpenAI(base_url=f"{SERVER_URL}/v1", api_key=VLLM_API_KEY)
model_list = client.models.list()
model_table = pd.DataFrame([model.to_dict() for model in model_list.data])
display(model_table[["id", "root", "max_model_len"]])

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">요청의 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">model</code>에는 모델 목록의 <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">id</code></strong>를 씁니다. 지금 실행한 서버의 이름은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">labor-sft</code>입니다.</p>
</div>

### 4-2. 질문 하나 보내기

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">노무 평가셋의 첫 질문을 보냅니다. <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">max_completion_tokens</code>는 답변으로 생성할 토큰 수의 상한</strong>이며, 이 실습에서는 256으로 제한합니다.</p>
</div>

In [ ]:
first_question = labor_eval_rows[0]["question"]
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": first_question},
]
response = client.chat.completions.create(
    model="labor-sft",
    messages=messages,
    temperature=0,
    max_completion_tokens=MAX_TOKENS,
)
print("질문:", first_question)
print("답변:", response.choices[0].message.content)

In [ ]:
response_info = {
    "종료 이유": response.choices[0].finish_reason,
    "입력 토큰 수": response.usage.prompt_tokens,
    "생성 토큰 수": response.usage.completion_tokens,
}
display(response_info)

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">stop</code>은 종료 토큰이나 정지 조건에 도달한 경우이고, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">length</code>는 길이 한도에 도달한 경우입니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">length</code>이면 필요한 답변이 끝까지 생성됐는지 확인합니다.</p>
</div>

### 4-3. 생성 상한을 낮춰 비교하기

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">같은 질문에서 생성 상한만 <strong style="font-size:1.13em;color:#1f3a5f;">8토큰</strong>으로 낮춥니다. 답변과 종료 이유가 어떻게 달라지는지 확인합니다.</p>
</div>

In [ ]:
short_response = client.chat.completions.create(
    model="labor-sft",
    messages=messages,
    temperature=0,
    max_completion_tokens=8,
)
print("답변:", short_response.choices[0].message.content)
print("종료 이유:", short_response.choices[0].finish_reason)

## 5. 여러 요청을 처리하는 내부 원리

### 5-1. KV 캐시의 역할

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">LLM은 <strong style="font-size:1.13em;color:#1f3a5f;">토큰을 하나씩 생성</strong>합니다. 다음 토큰을 고를 때도 앞에서 입력받고 생성한 문맥을 참고합니다.</p>
<p style="margin:6px 0;">아래는 <strong style="font-size:1.13em;color:#1f3a5f;">일반적인 전체 Attention</strong>의 설명입니다. 현재 위치의 <strong style="font-size:1.13em;color:#1f3a5f;">Q(Query)</strong>를 각 토큰의 <strong style="font-size:1.13em;color:#1f3a5f;">K(Key)</strong>와 비교해 관련도를 계산하고, <strong style="font-size:1.13em;color:#1f3a5f;">V(Value)</strong>를 모아 문맥을 반영합니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">KV 캐시</strong>는 이미 처리한 토큰의 K·V를 GPU 메모리에 저장한 것입니다. 다음 생성 단계에서는 <strong style="font-size:1.13em;color:#1f3a5f;">새로 처리할 토큰의 Q·K·V만 계산하고, 이전 K·V는 재사용</strong>합니다. 각 Attention 층에서 이런 저장과 재사용이 이루어집니다.</p>
</div>

<img style="display:block;max-width:100%;height:auto;margin:0;" src="images/14_kv_cache_flow.png" width="1050" alt="Prefill에서 입력의 K·V를 저장하고, Decode에서 새 K·V를 추가하며 이전 K·V를 읽습니다.">

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:6px 0;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">단계</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">처리하는 입력</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">캐시 사용</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><strong style="font-size:1.13em;color:#1f3a5f;">Prefill: 입력 처리</strong></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">프롬프트의 토큰들</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">입력 토큰의 K·V를 저장하고 첫 출력 토큰을 고릅니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><strong style="font-size:1.13em;color:#1f3a5f;">Decode: 반복 생성</strong></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">직전에 선택한 토큰 1개</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">새 K·V를 추가하고, 저장된 K·V를 읽어 다음 토큰을 고릅니다.</td>
</tr>
</tbody>
</table>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">KV 캐시가 줄이는 것은 이전 K·V를 다시 계산하는 일입니다.</strong> 저장된 문맥을 읽는 Attention 계산은 계속 필요합니다. 완성된 답변을 저장하는 캐시와는 역할이 다릅니다.</p>
</div>

### 5-2. GPU 메모리 구성

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">GPU 메모리(VRAM)는 모델 가중치뿐 아니라 <strong style="font-size:1.13em;color:#1f3a5f;">요청별 캐시와 중간 계산</strong>에도 쓰입니다.</p>
</div>

<img style="display:block;max-width:100%;height:auto;margin:0;" src="images/15_gpu_vram_structure.png" width="1050" alt="GPU 메모리에는 모델 가중치, 요청별 KV 캐시를 담는 풀, 중간 계산과 실행용 공간이 필요합니다.">

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">모델 가중치:</strong> 서버 시작 때 불러오며 여러 요청이 함께 사용합니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">KV 캐시:</strong> 각 요청이 처리한 문맥을 보관합니다. 일반적인 전체 Attention에서는 문맥이 길거나 동시 요청이 많을수록 필요한 공간도 늘어납니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">중간 계산·실행 공간:</strong> 계산 중 생기는 값, CUDA 연산과 그래프 등에 필요합니다.</li>
</ul>
<p style="margin:6px 0;">따라서 <strong style="font-size:1.13em;color:#1f3a5f;">모델이 GPU에 올라가는 것만으로 모든 요청을 받을 수 있는 것은 아닙니다</strong>. 요청을 처리할 캐시 공간도 남아 있어야 합니다.</p>
</div>

### 5-3. 블록 단위 배정과 PagedAttention

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">블록은 여러 토큰의 K·V를 담는 고정 크기 메모리 묶음</strong>입니다. vLLM은 서버 시작 때 확보한 캐시 공간을 블록 풀로 관리하고, 요청에 필요한 블록을 배정합니다.</p>
<p style="margin:6px 0;">예를 들어 <strong style="font-size:1.13em;color:#1f3a5f;">블록 하나에 16토큰의 K·V</strong>를 저장한다고 가정합니다. 아래 토큰 수는 <strong style="font-size:1.13em;color:#1f3a5f;">이미 처리한 입력과 출력의 합</strong>입니다.</p>
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:6px 0;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">처리한 토큰</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">필요한 블록</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">배정된 저장 칸</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">마지막 블록의 빈 칸</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">16개</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">1개</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">16칸</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">0칸</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">17개</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">2개</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">32칸</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">15칸</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">40개</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">3개</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">48칸</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">8칸</td>
</tr>
</tbody>
</table>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">17번째 토큰부터 두 번째 블록이 필요</strong>합니다. 처음부터 요청의 최대 길이 전체를 예약하지 않고, 처리할 토큰이 늘어날 때 블록을 추가합니다.</p>
</div>

<img style="display:block;max-width:100%;height:auto;margin:0;" src="images/01_paged_attention.png" width="1050" alt="요청의 논리 블록 0·1·2가 물리 블록 5·2·9에 연결됩니다.">

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">PagedAttention은 떨어져 있는 캐시 블록에서도 Attention을 계산하는 기법</strong>입니다. 요청의 토큰 순서인 <strong style="font-size:1.13em;color:#1f3a5f;">논리 블록</strong>과 GPU의 실제 저장 위치인 <strong style="font-size:1.13em;color:#1f3a5f;">물리 블록</strong>을 <strong style="font-size:1.13em;color:#1f3a5f;">블록 테이블</strong>로 연결합니다. 그림에서 요청 A는 논리 순서 0 → 1 → 2에 따라 물리 위치 5 → 2 → 9를 읽습니다. 세 블록이 나란히 있을 필요가 없습니다.</p>
<p style="margin:6px 0;">요청이 끝나면 그 요청이 사용하던 블록은 <strong style="font-size:1.13em;color:#1f3a5f;">다른 요청에 재배정할 수 있습니다</strong>. 서버가 확보한 GPU 메모리 전체를 운영체제에 돌려준다는 뜻은 아닙니다. 마지막 블록의 빈 칸은 남을 수 있지만, 최대 길이 전체를 예약할 때보다 낭비를 줄일 수 있습니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">그림은 원리 설명용입니다.</strong> 16토큰은 고정 규칙이 아닙니다. 실습의 Qwen3.5는 전체 Attention과 선형 Attention을 섞어 씁니다. 선형 Attention 층은 이전 문맥을 누적 상태로 관리합니다. 실제 캐시 구성과 블록 크기는 모델·엔진 설정에 따라 달라집니다. <a style="color:#245fa0;text-decoration:underline;" href="https://vllm.ai/blog/2023-06-20-vllm">PagedAttention 설명</a>, <a style="color:#245fa0;text-decoration:underline;" href="https://docs.vllm.ai/en/v0.27.0/design/hybrid_kv_cache_manager/">하이브리드 캐시 관리</a>를 참고합니다.</p>
</div>

### 5-4. 연속 배치로 계산 자리 채우기

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">정적 배치(static batching)</strong>는 묶음 전체가 끝난 뒤 다음 묶음을 시작합니다. 짧은 요청이 먼저 끝나면 그 계산 자리는 잠시 빕니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">연속 배치(continuous batching)</strong>는 생성 단계마다 완료된 요청을 빼고 대기 중인 요청을 넣습니다. 실행 여유가 생길 때 새 요청을 함께 처리해 GPU의 빈 시간을 줄입니다.</p>
</div>

<img style="display:block;max-width:100%;height:auto;margin:0;" src="images/02_continuous_batching.png" width="1050" alt="연속 배치는 A·C·D가 끝난 뒤 그 자리에 E·F·G 요청을 넣습니다.">

### 5-5. 시작 로그의 메모리 용량

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">tee</code>가 <strong style="font-size:1.13em;color:#1f3a5f;">RunPod의 <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">output/labor_server.log</code></strong>에 서버 로그를 저장했습니다. 이 파일을 내려받아 <strong style="font-size:1.13em;color:#1f3a5f;">내 PC의 단원 폴더 <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">output/labor_server.log</code></strong>에 저장한 뒤 실행합니다. <strong style="font-size:1.13em;color:#1f3a5f;">KV 캐시 용량과 최대 길이 기준의 동시 요청 추정치</strong>를 읽습니다.</p>
</div>

In [ ]:
log_text = (output_dir / "labor_server.log").read_text(encoding="utf-8")
for line in log_text.splitlines():
    if "GPU KV cache size" in line or "Maximum concurrency" in line:
        print(line)

In [ ]:
kv_match = re.search(r"GPU KV cache size: ([\d,]+) tokens", log_text)
kv_cache_tokens = int(kv_match.group(1).replace(",", ""))
concurrency_match = re.search(r"Maximum concurrency for [\d,]+ tokens per request: ([\d.]+)x", log_text)
max_concurrency = float(concurrency_match.group(1))
cache_info = {"KV 캐시(토큰)": kv_cache_tokens, "동시 요청 추정치": max_concurrency}
display(cache_info)

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Maximum concurrency</code>는 요청마다 최대 길이를 쓸 때의 <strong style="font-size:1.13em;color:#1f3a5f;">메모리 용량 추정치</strong>입니다. 실제 동시 처리 속도나 연결 수를 보장하는 값은 아닙니다.</p>
</div>

## 6. 여러 질문을 한꺼번에 보내기

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">LangChain의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ChatOpenAI</code>도 같은 서버를 부릅니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">batch</code>는 질문 묶음을 보내고 답변을 질문 순서대로 돌려줍니다.</p>
</div>

In [ ]:
labor_llm = ChatOpenAI(
    model="labor-sft", base_url=f"{SERVER_URL}/v1", api_key=VLLM_API_KEY,
    temperature=0, max_tokens=MAX_TOKENS,
)
eval_messages = []
for row in labor_eval_rows[:8]:
    eval_messages.append([("system", SYSTEM_PROMPT), ("human", row["question"])])
concurrent_answers = labor_llm.batch(eval_messages, config={"max_concurrency": 4})
answer_table = pd.DataFrame({
    "질문": [row["question"] for row in labor_eval_rows[:8]],
    "답변": [answer.content for answer in concurrent_answers],
})
display(answer_table)

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기:</strong> 질문 8개에 대한 답이 모두 표시되는지 확인합니다. 클라이언트의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">max_concurrency=4</code>는 동시에 보내는 요청 수의 상한이며 서버 내부 배치 설정과는 구분합니다.</p>
</div>

### 6-1. 서버 종료

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">RunPod에서 서버를 실행한 터미널</strong>로 돌아가 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">Ctrl+C</code></strong>를 누릅니다. 프롬프트가 돌아오면 같은 터미널에서 확인합니다.</p>
</div>

```bash
nvidia-smi
```

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">vLLM 서버 프로세스가 사라지고 GPU 메모리 사용량이 실행 전 수준으로 줄었는지</strong> 확인합니다. 서버 프로세스가 종료되면 모델 가중치와 KV 캐시 메모리도 반환됩니다. 확인을 마친 뒤 다음 단계로 이동합니다.</p>
</div>

## 7. 🖐️ 함께 따라하기: 전월세 상담 모델 서빙

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">같은 서빙 과정을 <strong style="font-size:1.13em;color:#1f3a5f;">전월세 상담 모델</strong>에 적용해 API로 답변을 받습니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">본실습과 다른 점</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">모델:</strong> 노무 모델 대신 SFT 단원의 전월세 상담 모델을 사용합니다. 서버 이름은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">lease-sft</code>입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">데이터:</strong> 전월세 시스템 메시지와 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">lease_eval.jsonl</code>의 앞 3개 질문을 사용합니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">설정:</strong> 서버 주소·최대 길이·생성 상한은 본실습과 같습니다.</li>
</ul>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">실습 진행 방법:</strong> 주석에 맞춰 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">...</code>를 채웁니다. 완성 코드는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">정답/교안_01_정답.ipynb</code>의 같은 절에 있습니다.</p>
</div>

### 7-1. 전월세 서버 실행과 데이터 준비

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">노무 서버 종료와 메모리 반환 확인을 마쳤다면 같은 RunPod 터미널에서 다음 명령을 실행합니다. PC의 공개 주소·API 키는 그대로 사용합니다.</p>
</div>

**복사해서 실행 · RunPod 터미널**

```bash
uv run vllm serve muahseol/enkoa-qwen3.5-0.8b-lease-sft \
  --served-model-name lease-sft \
  --dtype bfloat16 \
  --max-model-len 4096 \
  --reasoning-parser qwen3 \
  --default-chat-template-kwargs '{"enable_thinking": false}' \
  --host 0.0.0.0 \
  --port 8000 \
  --api-key "$VLLM_API_KEY" 2>&1 | tee output/lease_server.log
```

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">로그에 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">Application startup complete</code></strong>가 나오면 요청을 보낼 수 있습니다. RunPod 터미널은 그대로 두고 <strong style="font-size:1.13em;color:#1f3a5f;">내 PC의 노트북</strong>으로 돌아옵니다. 첫 실행은 모델 다운로드와 GPU 계산 준비에 시간이 걸릴 수 있습니다.</p>
</div>

In [ ]:
SYSTEM_PROMPT_LEASE = "너는 전월세 상담 챗봇이다. 주택임대차보호법 조문에 근거해 존댓말로 간결히 답한다."
lease_eval_rows = load_jsonl(day70_dir / "data" / "따라하기" / "lease_eval.jsonl")

In [ ]:
lease_question_table = pd.DataFrame(lease_eval_rows[:3])[['question', 'reference_answer']]
display(lease_question_table)

### 7-2. API 연결과 질문 하나 보내기

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">작성할 것:</strong> (1) <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">SERVER_URL</code>의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/v1</code> 주소에 연결할 클라이언트, (2) <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">lease-sft</code>에 보낼 요청을 작성합니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">temperature=0</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">max_completion_tokens=MAX_TOKENS</code>를 사용합니다.</p>
</div>

<details><summary>힌트</summary>

```text
접근방법: 교안 01의 4-1·4-2를 참고합니다.
OpenAI 클라이언트의 base_url에 서버 주소를 지정합니다. 요청에는 전월세 서버 이름과 아래 lease_messages를 사용합니다.
```

</details>

In [ ]:
# TODO (1): vLLM 서버에 연결하는 OpenAI 클라이언트를 만듭니다.
lease_client = ...
lease_messages = [
    {"role": "system", "content": SYSTEM_PROMPT_LEASE},
    {"role": "user", "content": lease_eval_rows[0]["question"]},
]
# TODO (2): 전월세 모델에 질문을 보냅니다.
lease_response = ...
print("답변:", lease_response.choices[0].message.content)
print("종료 이유:", lease_response.choices[0].finish_reason)

### 7-3. 여러 질문 보내고 서버 종료하기

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">작성할 것:</strong> (3) 전월세 모델을 호출하는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ChatOpenAI</code> 클라이언트, (4) 앞 3개 질문을 보내는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">batch</code> 요청을 작성합니다. 생성 조건은 본실습과 같고 동시 요청 상한은 3입니다.</p>
</div>

<details><summary>힌트</summary>

```text
접근방법: 교안 01의 6을 참고합니다.
ChatOpenAI에도 같은 서버 주소와 모델 이름을 지정합니다. batch에는 아래 lease_message_lists를 넘기고 config에 max_concurrency를 지정합니다.
```

</details>

In [ ]:
# TODO (3): 전월세 모델을 호출하는 LangChain 클라이언트를 만듭니다.
lease_llm = ...
lease_message_lists = []
for row in lease_eval_rows[:3]:
    lease_message_lists.append([("system", SYSTEM_PROMPT_LEASE), ("human", row["question"])])
# TODO (4): 질문 세 개를 한꺼번에 보냅니다.
lease_answers = ...
lease_answer_table = pd.DataFrame({
    "질문": [row["question"] for row in lease_eval_rows[:3]],
    "정답 답변": [row["reference_answer"] for row in lease_eval_rows[:3]],
    "모델 답변": [answer.content for answer in lease_answers],
})
display(lease_answer_table)

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 확인:</strong> 세 질문의 답변이 표시되는지 확인합니다. 정답 답변과 표현이 달라도 핵심 내용이 맞는지 읽어 봅니다. 앞의 단일 요청에서 종료 이유가 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">length</code>였다면 답변이 잘렸는지도 확인합니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">RunPod에서 서버를 실행한 터미널</strong>로 돌아가 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">Ctrl+C</code></strong>를 누릅니다. 프롬프트가 돌아오면 같은 터미널에서 확인합니다.</p>
</div>

```bash
nvidia-smi
```

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">vLLM 서버 프로세스가 사라지고 GPU 메모리 사용량이 실행 전 수준으로 줄었는지</strong> 확인합니다. 서버 프로세스가 종료되면 모델 가중치와 KV 캐시 메모리도 반환됩니다. 확인을 마친 뒤 다음 단계로 이동합니다.</p>
</div>

## 8. 핵심 정리

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">vLLM:</strong> 직접 실행하는 LLM을 OpenAI 호환 API로 제공합니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">서버 실행·종료:</strong> day75 폴더에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">uv run vllm serve</code>를 실행하고 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Ctrl+C</code>로 종료합니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">API 요청:</strong> 서버 주소·모델 이름·메시지·생성 상한을 지정합니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">PagedAttention·연속 배치:</strong> 각각 KV 캐시 공간과 여러 요청의 계산을 효율적으로 사용합니다.</li>
</ul>
</div>